# Notebook 01: Dataset Ingestion, Quality Audit & ML-Readiness Inspection

**Project**: Student Performance Classification using Classical Machine Learning  
**Objective**: Perform a rigorous, read-only exploratory audit of the raw dataset (`data/student_performance_data.csv`) without model training, data modification, or synthetic data generation.

### 1. Import Required Classical ML / Data Inspection Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_PATH = os.path.join('..', 'data', 'student_performance_data.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('data', 'student_performance_data.csv')

print(f"Target dataset path: {DATA_PATH}")

### 2. Task 1: Load Dataset & Inspect Basic Structure

In [ ]:
# Load raw dataset without modifying original file
df = pd.read_csv(DATA_PATH)

print(f"Dataset Dimensions (Rows, Columns): {df.shape}")
print(f"Total Records: {len(df)}")
print(f"Columns: {list(df.columns)}")

df.info()

In [ ]:
# Inspect first 5 records
df.head()

In [ ]:
# Inspect last 5 records
df.tail()

### 3. Task 2 & 6: Data Quality Audit (Missing Values, Duplicates, Unique Counts)

In [ ]:
# Missing values audit
null_counts = df.isnull().sum()
null_pcts = (null_counts / len(df)) * 100

quality_audit_df = pd.DataFrame({
    'Data_Type': df.dtypes,
    'Unique_Values': df.nunique(),
    'Null_Count': null_counts,
    'Null_Percentage (%)': null_pcts.round(2)
})

print(f"Total missing cells across entire dataset: {df.isnull().sum().sum()}")
print(f"Total duplicate rows: {df.duplicated().sum()}")
quality_audit_df

In [ ]:
# Inspect sample records containing missing values
df[df.isnull().any(axis=1)].head(10)

### 4. Task 8: Numerical Feature Summary & Range Validation

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

stats_df = pd.DataFrame({
    'Count': df[num_cols].count(),
    'Mean': df[num_cols].mean().round(4),
    'Std_Dev': df[num_cols].std().round(4),
    'Min': df[num_cols].min(),
    '25% (Q1)': df[num_cols].quantile(0.25),
    'Median (Q2)': df[num_cols].median(),
    '75% (Q3)': df[num_cols].quantile(0.75),
    'Max': df[num_cols].max()
})

stats_df

In [ ]:
# Verify that all numerical score and percentage ranges are within expected [0, 100]
for col in num_cols:
    in_range = (df[col].min() >= 0.0) and (df[col].max() <= 100.0)
    print(f"Column '{col}': Range [{df[col].min()}, {df[col].max()}] -> Valid Domain [0, 100]: {in_range}")

### 5. Task 3 & 7: Target Variable Inspection & Class Distribution

In [ ]:
target_col = 'Performance_Category'

counts = df[target_col].value_counts(dropna=False)
percentages = (df[target_col].value_counts(normalize=True, dropna=False) * 100).round(2)

target_dist = pd.DataFrame({
    'Sample_Count': counts,
    'Proportion (%)': percentages
})

print(f"Target Column: {target_col}")
print(f"Target Unique Classes: {list(df[target_col].unique())}")
print(f"Class Imbalance Ratio (Majority / Minority): {counts.max() / counts.min():.2f}")
target_dist

### 6. Task 5: Target Leakage Investigation (Performance_Score vs. Performance_Category)

In [ ]:
# Inspect Performance_Score distribution per target class
score_by_cat = df.groupby('Performance_Category')['Performance_Score'].agg(['count', 'min', 'max', 'mean', 'median', 'std'])
score_by_cat

In [ ]:
# Test linear reconstruction of Performance_Score from raw features
clean_subset = df.dropna()
X_features = clean_subset[['MST_Score', 'Quiz_Score', 'Attendance_Percent', 'Assignment_Score']]
y_score = clean_subset['Performance_Score']

weights, residuals, rank, s = np.linalg.lstsq(X_features, y_score, rcond=None)

print("Determined Weights for Performance_Score:")
for name, w in zip(X_features.columns, weights):
    print(f"  {name}: {w:.4f}")
print(f"\nResidual Sum of Squares: {residuals[0]:.2e}")

# Verification:
reconstructed = (0.40 * clean_subset['MST_Score'] + 
                 0.20 * clean_subset['Quiz_Score'] + 
                 0.20 * clean_subset['Attendance_Percent'] + 
                 0.20 * clean_subset['Assignment_Score'])
max_err = np.max(np.abs(clean_subset['Performance_Score'] - reconstructed))
print(f"Max difference between Performance_Score and exact 40/20/20/20 formula: {max_err:.8f}")

### 7. Audit Summary & Decision

1. **Input Features for $X$**: `MST_Score`, `Quiz_Score`, `Attendance_Percent`, `Assignment_Score` (4 numerical features).
2. **Target for $y$**: `Performance_Category` (3 classes: `Needs Improvement`, `Average Performer`, `High Performer`).
3. **Excluded Columns**:
   - `Student_ID`: Identifier with no predictive relationship.
   - `Performance_Score`: 100% Target Leakage (deterministic predecessor of `Performance_Category`).
4. **Missing Values**: 24 entries (0.6% per feature) to be handled via `SimpleImputer(strategy='median')` in pipeline.
5. **Stratification**: Necessary during train/test split due to 8.6% minority class representation.
6. **Status**: **Dataset is verified and ready for Phase 2B (Preprocessing & Pipeline Design).**